In [1]:
import os
import glob
from datasets import load_dataset

# ===== HuggingFace Cache Configuration =====
cwd = os.getcwd()
project_root = os.path.abspath(os.path.join(cwd, "..")) if cwd.endswith("trains") else cwd
HF_CACHE_DIR = os.path.join(project_root, "hf_cache")

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_CACHE_DIR, "datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print(f"[✓] HF_CACHE_DIR = {HF_CACHE_DIR}")


[✓] HF_CACHE_DIR = /lustrefs/disk/project/pv915002-hpcign/tts-stt/hf_cache


In [2]:
from unsloth import FastLanguageModel
import torch
import glob

# ค้นหาตำแหน่ง snapshot ของโมเดลใน hf_cache
model_paths = glob.glob(os.path.join(HF_CACHE_DIR, "hub", "models--unsloth--orpheus-3b-0.1-ft", "snapshots", "*"))
MODEL_PATH = model_paths[0] if model_paths else "unsloth/orpheus-3b-0.1-ft"
print(f"[✓] Loading model from local path: {MODEL_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_PATH,
    max_seq_length = 2048, # Choose any for long context!
    dtype = None, # Select None for auto detection
    load_in_4bit = False, # Select True for 4bit which reduces memory usage
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[fla.utils._device|WARNING]Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
[✓] Loading model from local path: /lustrefs/disk/project/pv915002-hpcign/tts-stt/hf_cache/hub/models--unsloth--orpheus-3b-0.1-ft/snapshots/eae2b6e5e429c81b95ac42a883ac64f126583d43
==((====))==  Unsloth 2026.8.8: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.496 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 64,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.8.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


<a name="Data"></a>
### Data Prep  

We will use the `Etherll/kaira`, which is designed for training TTS models. Ensure that your dataset follows the required format: **text, audio** for single-speaker models or **source, text, audio** for multi-speaker models. You can modify this section to accommodate your own dataset, but maintaining the correct structure is essential for optimal training.

In [9]:
from datasets import load_dataset
import glob

# โหลดไฟล์ parquet จาก local hf_cache ตรงๆ โดยไม่ต้องออกอินเทอร์เน็ต
data_files = glob.glob(os.path.join(HF_CACHE_DIR, "datasets", "datasets--Thanarit--Thai-Voice-Test7", "**", "*.parquet"), recursive=True)
if not data_files:
    data_files = glob.glob(os.path.join(HF_CACHE_DIR, "**", "*.parquet"), recursive=True)

print(f"[✓] Loading {len(data_files)} parquet file(s) from disk: {data_files}")
dataset = load_dataset("parquet", data_files=data_files, split="train[:1500]")
dataset = dataset.rename_column("transcript", "text")

[✓] Loading 1 parquet file(s) from disk: ['/lustrefs/disk/project/pv915002-hpcign/tts-stt/hf_cache/datasets/datasets--Thanarit--Thai-Voice-Test7/snapshots/ede9064471fa5da1686762ac0a3682b97dd23609/data/train/shard_00000.parquet']


In [10]:
dataset

Dataset({
    features: ['ID', 'speaker_id', 'Language', 'audio', 'text', 'length', 'dataset_name', 'confidence_score'],
    num_rows: 1500
})

In [11]:
#@title Tokenization Function

import locale
import os
import torch
import glob
from snac import SNAC
import datasets

# ============================================================
# 1. Audio / Torchaudio setup
# ============================================================

try:
    import torchaudio.transforms as T
    USE_TORCHAUDIO = True
except Exception as e:
    print(f"[!] torchaudio load notice: {e}")
    print("[✓] Using PyTorch native 1D resampler fallback.")
    USE_TORCHAUDIO = False

locale.getpreferredencoding = lambda: "UTF-8"

# ============================================================
# 2. Dataset Audio -> 24kHz
# ============================================================

dataset = dataset.cast_column(
    "audio",
    datasets.Audio(sampling_rate=24000)
)

ds_sample_rate = dataset.features["audio"].sampling_rate
print(f"[✓] Dataset sample rate: {ds_sample_rate} Hz")

# ============================================================
# 3. Load SNAC
# ============================================================

snac_paths = glob.glob(
    os.path.join(
        HF_CACHE_DIR,
        "hub",
        "models--hubertsiuzdak--snac_24khz",
        "snapshots",
        "*"
    )
)

SNAC_PATH = (
    snac_paths[0]
    if snac_paths
    else "hubertsiuzdak/snac_24khz"
)

print(f"[✓] Loading SNAC Audio Codec from: {SNAC_PATH}")

snac_model = (
    SNAC.from_pretrained(SNAC_PATH)
    .eval()
    .to("cuda")
)

# ============================================================
# 4. Audio limits
# ============================================================

TARGET_SAMPLE_RATE = 24000
MAX_AUDIO_SECONDS = 15.0
MAX_AUDIO_SAMPLES = int(
    TARGET_SAMPLE_RATE * MAX_AUDIO_SECONDS
)
MAX_SEQ_LENGTH = 2048
SNAC_TOKENS_PER_FRAME = 7

# ============================================================
# 5. Audio -> SNAC tokens
# ============================================================

def tokenise_audio(waveform):
    waveform = torch.from_numpy(
        waveform
    ).unsqueeze(0).to(dtype=torch.float32)

    if ds_sample_rate != TARGET_SAMPLE_RATE:
        if USE_TORCHAUDIO:
            resample_transform = T.Resample(
                orig_freq=ds_sample_rate,
                new_freq=TARGET_SAMPLE_RATE
            )
            waveform = resample_transform(waveform)
        else:
            target_len = int(
                waveform.shape[-1]
                * (
                    TARGET_SAMPLE_RATE
                    / ds_sample_rate
                )
            )
            waveform = torch.nn.functional.interpolate(
                waveform.unsqueeze(0),
                size=target_len,
                mode="linear",
                align_corners=False
            ).squeeze(0)

    waveform = waveform.unsqueeze(0).to("cuda")

    with torch.inference_mode():
        codes = snac_model.encode(waveform)

    all_codes = []
    for i in range(codes[0].shape[1]):
        all_codes.append(
            codes[0][0][i].item()
            + 128266
        )
        all_codes.append(
            codes[1][0][2 * i].item()
            + 128266
            + 4096
        )
        all_codes.append(
            codes[2][0][4 * i].item()
            + 128266
            + (2 * 4096)
        )
        all_codes.append(
            codes[2][0][(4 * i) + 1].item()
            + 128266
            + (3 * 4096)
        )
        all_codes.append(
            codes[1][0][(2 * i) + 1].item()
            + 128266
            + (4 * 4096)
        )
        all_codes.append(
            codes[2][0][(4 * i) + 2].item()
            + 128266
            + (5 * 4096)
        )
        all_codes.append(
            codes[2][0][(4 * i) + 3].item()
            + 128266
            + (6 * 4096)
        )

    return all_codes

# ============================================================
# 6. Convert audio -> codes
# ============================================================

def add_codes(example):
    codes_list = None
    try:
        answer_audio = example.get("audio")
        if (
            answer_audio
            and "array" in answer_audio
        ):
            audio_array = answer_audio["array"]
            if len(audio_array) > MAX_AUDIO_SAMPLES:
                example["codes_list"] = None
                return example
            codes_list = tokenise_audio(
                audio_array
            )
    except Exception as e:
        print(
            f"Skipping row due to error: {e}"
        )
        codes_list = None

    example["codes_list"] = codes_list
    return example

# ============================================================
# 7. Generate SNAC codes
# ============================================================

print("[✓] Generating SNAC codes...")
dataset = dataset.map(
    add_codes,
    remove_columns=["audio"]
)

# ============================================================
# 8. Remove invalid samples
# ============================================================

dataset = dataset.filter(
    lambda x:
        x["codes_list"] is not None
)
dataset = dataset.filter(
    lambda x:
        len(x["codes_list"]) > 0
)
print(
    f"[✓] Valid samples after audio filtering: "
    f"{len(dataset)}"
)

# ============================================================
# 9. Remove duplicate SNAC frames
# ============================================================

def remove_duplicate_frames(example):
    vals = example["codes_list"]
    if len(vals) % 7 != 0:
        raise ValueError(
            "Input list length must be divisible by 7"
        )
    result = vals[:7]
    removed_frames = 0
    for i in range(
        7,
        len(vals),
        7
    ):
        current_first = vals[i]
        previous_first = result[-7]
        if current_first != previous_first:
            result.extend(
                vals[i:i + 7]
            )
        else:
            removed_frames += 1
    example["codes_list"] = result
    return example

dataset = dataset.map(
    remove_duplicate_frames
)

# ============================================================
# 10. Token IDs
# ============================================================

tokeniser_length = 128256
start_of_text = 128000
end_of_text = 128009

start_of_speech = tokeniser_length + 1
end_of_speech = tokeniser_length + 2
start_of_human = tokeniser_length + 3
end_of_human = tokeniser_length + 4
start_of_ai = tokeniser_length + 5
end_of_ai = tokeniser_length + 6
pad_token = tokeniser_length + 7
audio_tokens_start = tokeniser_length + 10

# ============================================================
# 11. Prompt information
# ============================================================

tok_info = """
*** HERE you can modify the text prompt

Single-speaker:
    f"{example['text']}"

Multi-speaker:
    f"{example['source']}: {example['text']}"
"""
print(tok_info)

# ============================================================
# 12. Create input_ids
# ============================================================

def create_input_ids(example):
    if "source" in example:
        text_prompt = (
            f"{example['source']}: "
            f"{example['text']}"
        )
    else:
        text_prompt = example["text"]

    text_ids = tokenizer.encode(
        text_prompt,
        add_special_tokens=True
    )
    text_ids.append(
        end_of_text
    )
    example["text_tokens"] = text_ids

    prefix = (
        [start_of_human]
        + text_ids
        + [end_of_human]
        + [start_of_ai]
        + [start_of_speech]
    )

    suffix = [
        end_of_speech,
        end_of_ai
    ]

    codes = example["codes_list"]
    available_audio_tokens = (
        MAX_SEQ_LENGTH
        - len(prefix)
        - len(suffix)
    )
    max_audio_tokens = (
        available_audio_tokens // 7
    ) * 7

    if max_audio_tokens <= 0:
        example["input_ids"] = []
        example["labels"] = []
        example["attention_mask"] = []
        return example

    codes = codes[
        :max_audio_tokens
    ]

    input_ids = (
        prefix
        + codes
        + suffix
    )

    if len(input_ids) > MAX_SEQ_LENGTH:
        input_ids = input_ids[
            :MAX_SEQ_LENGTH
        ]
        audio_start = len(prefix)
        audio_length = (
            len(input_ids)
            - audio_start
            - len(suffix)
        )
        audio_length = (
            audio_length // 7
        ) * 7
        input_ids = (
            prefix
            + codes[:audio_length]
            + suffix
        )

    example["input_ids"] = input_ids
    example["labels"] = input_ids.copy()
    example["attention_mask"] = [
        1
    ] * len(input_ids)
    return example

# ============================================================
# 13. Generate final training tensors
# ============================================================

dataset = dataset.map(
    create_input_ids,
    remove_columns=[
        "text",
        "codes_list"
    ]
)

# ============================================================
# 14. Remove samples with invalid sequence length
# ============================================================

dataset = dataset.filter(
    lambda x:
        len(x["input_ids"]) > 0
)
dataset = dataset.filter(
    lambda x:
        len(x["input_ids"])
        <= MAX_SEQ_LENGTH
)

# ============================================================
# 15. Remove unnecessary columns
# ============================================================

columns_to_keep = [
    "input_ids",
    "labels",
    "attention_mask"
]
columns_to_remove = [
    col
    for col in dataset.column_names
    if col not in columns_to_keep
]
dataset = dataset.remove_columns(
    columns_to_remove
)

# ============================================================
# 16. FINAL DATASET VALIDATION
# ============================================================

print()
print("=" * 60)
print("DATASET VALIDATION")
print("=" * 60)

lengths = [
    len(x["input_ids"])
    for x in dataset
]

print(
    f"Samples              : {len(dataset)}"
)
print(
    f"Min sequence length  : {min(lengths)}"
)
print(
    f"Max sequence length  : {max(lengths)}"
)
print(
    f"Max allowed          : {MAX_SEQ_LENGTH}"
)

assert max(lengths) <= MAX_SEQ_LENGTH

for x in dataset:
    assert (
        len(x["input_ids"])
        == len(x["labels"])
    )
    assert (
        len(x["input_ids"])
        == len(x["attention_mask"])
    )

print()
print("✓ input_ids <= 2048")
print("✓ labels == input_ids")
print("✓ attention_mask == input_ids")
print("✓ Dataset validation passed")
print("=" * 60)


[!] torchaudio load notice: Could not load this library: /lustrefs/disk/project/pv915002-hpcign/env_test/lib/python3.10/site-packages/torchaudio/lib/_torchaudio.abi3.so
[✓] Using PyTorch native 1D resampler fallback.
[✓] Dataset sample rate: 24000 Hz
[✓] Loading SNAC Audio Codec from: /lustrefs/disk/project/pv915002-hpcign/tts-stt/hf_cache/hub/models--hubertsiuzdak--snac_24khz/snapshots/d73ad176a12188fcf4f360ba3bf2c2fbbe8f58ec
[✓] Generating SNAC codes...


Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1479 [00:00<?, ? examples/s]

[✓] Valid samples after audio filtering: 1479


...


*** HERE you can modify the text prompt

Single-speaker:
    f"{example['text']}"

Multi-speaker:
    f"{example['source']}: {example['text']}"



Map:   0%|          | 0/1479 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1479 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1479 [00:00<?, ? examples/s]


DATASET VALIDATION
Samples              : 1479
Min sequence length  : 104
Max sequence length  : 1381
Max allowed          : 2048

✓ input_ids <= 2048
✓ labels == input_ids
✓ attention_mask == input_ids
✓ Dataset validation passed


In [12]:
dataset

Dataset({
    features: ['input_ids', 'labels', 'attention_mask'],
    num_rows: 1479
})

<a name="Train"></a>
### Train the model
Now let's use Hugging Face `Trainer`! More docs here: [Transformers docs](https://huggingface.co/docs/transformers/main_classes/trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

**Note:** Using a per_device_train_batch_size >1 may lead to errors if multi-GPU setup to avoid issues, ensure CUDA_VISIBLE_DEVICES is set to a single GPU (e.g., CUDA_VISIBLE_DEVICES=0).

In [13]:
from transformers import TrainingArguments,Trainer,DataCollatorForSeq2Seq
trainer = Trainer(
    model = model,
    train_dataset = dataset,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 5, # Set this for 1 full training run.
        # max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
5.713 GB of memory reserved.


In [14]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,479 | Num Epochs = 5 | Total steps = 1,850
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 97,255,424 of 3,398,122,496 (2.86% trained)


Step,Training Loss
1,5.227300
2,4.961300
3,4.945000
4,4.855400
5,4.729600
6,4.789200
7,4.743800
8,4.918300
9,4.795200
10,4.843900


KeyboardInterrupt: 

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Inference"></a>
### Inference
Let's run the model! You can change the prompts

In [15]:
prompts = [
    "สวัสดีครับ ผมทดสอบพูดภาษาไทย",
]

chosen_voice = None # None for single-speaker

In [16]:
#@title Run Inference


FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Moving snac_model cuda to cpu
snac_model.to("cpu")

prompts_ = [(f"{chosen_voice}: " + p) if chosen_voice else p for p in prompts]

all_input_ids = []

for prompt in prompts_:
  input_ids = tokenizer(prompt, return_tensors = "pt").input_ids
  all_input_ids.append(input_ids)

start_token = torch.tensor([[ 128259]], dtype = torch.int64) # Start of human
end_tokens = torch.tensor([[128009, 128260]], dtype = torch.int64) # End of text, End of human

all_modified_input_ids = []
for input_ids in all_input_ids:
  modified_input_ids = torch.cat([start_token, input_ids, end_tokens], dim = 1) # SOH SOT Text EOT EOH
  all_modified_input_ids.append(modified_input_ids)

all_padded_tensors = []
all_attention_masks = []
max_length = max([modified_input_ids.shape[1] for modified_input_ids in all_modified_input_ids])
for modified_input_ids in all_modified_input_ids:
  padding = max_length - modified_input_ids.shape[1]
  padded_tensor = torch.cat([torch.full((1, padding), 128263, dtype = torch.int64), modified_input_ids], dim = 1)
  attention_mask = torch.cat([torch.zeros((1, padding), dtype = torch.int64), torch.ones((1, modified_input_ids.shape[1]), dtype = torch.int64)], dim = 1)
  all_padded_tensors.append(padded_tensor)
  all_attention_masks.append(attention_mask)

all_padded_tensors = torch.cat(all_padded_tensors, dim = 0)
all_attention_masks = torch.cat(all_attention_masks, dim = 0)

input_ids = all_padded_tensors.to("cuda")
attention_mask = all_attention_masks.to("cuda")
generated_ids = model.generate(
      input_ids = input_ids,
      attention_mask = attention_mask,
      max_new_tokens = 1200,
      do_sample = True,
      temperature = 0.6,
      top_p = 0.95,
      repetition_penalty = 1.1,
      num_return_sequences = 1,
      eos_token_id = 128258,
     use_cache = True
  )
token_to_find = 128257
token_to_remove = 128258

token_indices = (generated_ids == token_to_find).nonzero(as_tuple = True)

if len(token_indices[1]) > 0:
    last_occurrence_idx = token_indices[1][-1].item()
    cropped_tensor = generated_ids[:, last_occurrence_idx+1:]
else:
    cropped_tensor = generated_ids

mask = cropped_tensor != token_to_remove

processed_rows = []

for row in cropped_tensor:
    masked_row = row[row != token_to_remove]
    processed_rows.append(masked_row)

code_lists = []

for row in processed_rows:
    row_length = row.size(0)
    new_length = (row_length // 7) * 7
    trimmed_row = row[:new_length]
    trimmed_row = [t - 128266 for t in trimmed_row]
    code_lists.append(trimmed_row)


def redistribute_codes(code_list):
  layer_1 = []
  layer_2 = []
  layer_3 = []
  for i in range((len(code_list)+1)//7):
    layer_1.append(code_list[7*i])
    layer_2.append(code_list[7*i+1]-4096)
    layer_3.append(code_list[7*i+2]-(2*4096))
    layer_3.append(code_list[7*i+3]-(3*4096))
    layer_2.append(code_list[7*i+4]-(4*4096))
    layer_3.append(code_list[7*i+5]-(5*4096))
    layer_3.append(code_list[7*i+6]-(6*4096))
  codes = [torch.tensor(layer_1).unsqueeze(0),
         torch.tensor(layer_2).unsqueeze(0),
         torch.tensor(layer_3).unsqueeze(0)]

  # codes = [c.to("cuda") for c in codes]
  audio_hat = snac_model.decode(codes)
  return audio_hat

my_samples = []
for code_list in code_lists:
  samples = redistribute_codes(code_list)
  my_samples.append(samples)
from IPython.display import display, Audio
if len(prompts) != len(my_samples):
  raise Exception("Number of prompts and samples do not match")
else:
  for i in range(len(my_samples)):
    print(prompts[i])
    samples = my_samples[i]
    display(Audio(samples.detach().squeeze().to("cpu").numpy(), rate = 24000))
# Clean up to save RAM
del my_samples,samples

สวัสดีครับ ผมทดสอบพูดภาษาไทย


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("orpheus_lora")  # Local saving
tokenizer.save_pretrained("orpheus_lora")
# model.push_to_hub("your_name/orpheus_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/orpheus_lora", token = "YOUR_HF_TOKEN") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

### Saving to float16

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("orpheus_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/orpheus_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("orpheus_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/orpheus_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("orpheus_lora")
    tokenizer.save_pretrained("orpheus_lora")
if False:
    model.push_to_hub("HF_USERNAME/orpheus_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/orpheus_lora", token = "YOUR_HF_TOKEN")

Unsloth: You have 1 CPUs. Using `safe_serialization` is 10x slower.
We shall switch to Pytorch saving, which might take 3 minutes and not 30 minutes.
To force `safe_serialization`, set it to `None` instead.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 15.1G

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 3.99 out of 12.67 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...

100%|██████████| 28/28 [00:01<00:00, 27.83it/s]

Unsloth: Saving tokenizer... Done.
Unsloth: Saving model/pytorch_model-00001-of-00002.bin...
Unsloth: Saving model/pytorch_model-00002-of-00002.bin...
Done.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)
</div>